# Live demo — Prediction + XAI (LayerCAM, GradCAM++, Occlusion)


## 0. Setup

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/USTH-B3-Projects/DL2026-Group5-Project30.git'
    REPO_DIR = '/content/DL2026-Group5-Project30'

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

    !pip install -q datasets grad-cam captum gdown

print('Working directory:', os.getcwd())

## 1. Download checkpoint

In [ ]:
import gdown
from pathlib import Path

from src.training.config import CHECKPOINT_DIR

CHECKPOINT_NAME = 'resnet50_warmup_seed2026_best.pt'
CHECKPOINT_FILE_ID = '1pke61YPrQXScSIqSPTMKHnL7r8Tpr9r9'   # from the Drive share link

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
checkpoint_dest = Path(CHECKPOINT_DIR) / CHECKPOINT_NAME
gdown.download(id=CHECKPOINT_FILE_ID, output=str(checkpoint_dest), quiet=False)
print('Saved to', checkpoint_dest)

## 2. Import

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

from src.training.config import DEVICE
from src.xai.common import load_model, denormalize
from src.xai.cam_methods import run_layercam, run_gradcam_plusplus
from src.xai.occlusion import run_occlusion

print('Device:', DEVICE)

## 3. Load the selected final model: ResNet-50 (seed 2026)

In [ ]:
MODEL_NAME = 'resnet50_warmup'
CLASS_NAMES = ['normal', 'early', 'advanced']

model, target_layers = load_model(CHECKPOINT_NAME)
print(f'Loaded {CHECKPOINT_NAME}')

## 4. Load the demo images

In [ ]:
import random

DEMO_MODE = "all"
DEMO_IMAGE_NAME = None

DEMO_DIR = Path('demo_samples')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

EVAL_IMAGE_SIZE = 320
eval_transform = transforms.Compose([
    transforms.Resize((EVAL_IMAGE_SIZE, EVAL_IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]),
])

def _local_image_paths():
    paths = sorted(p for p in DEMO_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    if not paths:
        raise FileNotFoundError(f'No images found in {DEMO_DIR}. Add a few representative fundus images '
                                f'(.jpg/.jpeg/.png) there before running this notebook.')
    return paths

demo_samples = []   # list of (label, img_tensor) where label is a file name

if DEMO_MODE == 'all':
    for path in _local_image_paths():
        image = Image.open(path).convert('RGB')
        demo_samples.append((path.name, eval_transform(image)))

elif DEMO_MODE == 'random':
    path = random.choice(_local_image_paths())
    image = Image.open(path).convert('RGB')
    demo_samples.append((path.name, eval_transform(image)))

elif DEMO_MODE == 'named':
    if not DEMO_IMAGE_NAME:
        raise ValueError('Set DEMO_FILE_NAME to a file name in demo_samples/')
    
    path = DEMO_DIR / DEMO_IMAGE_NAME
    if not path.exists():
        raise FileNotFoundError(f'File {path} does not exist.')
    
    image = Image.open(path).convert('RGB')
    demo_samples.append((path.name, eval_transform(image)))

else:
    raise ValueError(f'Unknown DEMO_MODE: {DEMO_MODE}. Must be one of "all", "random", "named".')

print(f'Loaded {len(demo_samples)} demo image(s):')
for label, _ in demo_samples:
    print(' ', label)

## 5. Predict + run all 3 XAI methods on each demo image

In [ ]:
method_names = ['Original', 'LayerCAM', 'GradCAM++', 'Occlusion']

for label, img_tensor in demo_samples:
    input_tensor = img_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred_label = probs.argmax().item()

    rgb_float = denormalize(img_tensor)
    layercam_vis = run_layercam(model, target_layers, input_tensor, pred_label, rgb_float)
    gradcampp_vis = run_gradcam_plusplus(model, target_layers, input_tensor, pred_label, rgb_float)
    occlusion_vis = run_occlusion(model, input_tensor, pred_label, rgb_float)

    panels = [rgb_float, layercam_vis, gradcampp_vis, occlusion_vis]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, panel, name in zip(axes, panels, method_names):
        ax.imshow(panel)
        ax.axis('off')
        ax.set_title(name)

    probs_str = ', '.join(f'{c}={p:.2f}' for c, p in zip(CLASS_NAMES, probs.tolist()))
    fig.suptitle(f'{label}  ->  predicted: {CLASS_NAMES[pred_label]}  ({probs_str})')
    fig.tight_layout()
    plt.show()